## Compile DupCaller burden

This notebook is used to compile DupCaller burden estimates from the output of the tool.

In [4]:
# --- 1. Setup: imports, cohort configuration, output paths ---
from pathlib import Path
import pandas as pd

# Extract burden results
dupcaller_path = "/data/bbg/nobackup2/prominent/duplex_seq_tests/error_rate/cord_blood/bbg/2026-03-dupcaller"

# IDT samples
dupcaller_idt_samples = [
        "SC001_B1_1_H_2",
        "SC002_B1_1_H_2",
        "SC003_B1_1_H_2",
    ]

# Row offset per protocol: observed, CI_low, CI_high are consecutive
# File layout: rows 0-2 uncorrected, 4-6 corrected, 8=genome coverage, 9-11 unmasked
PROTOCOL_ROWS = {"uncorrected": 0, "corrected": 4, "unmasked": 9}

# Suffix appended to the sample name per protocol to keep entries unique
PROTOCOL_SUFFIX = {"corrected": "_DC_corr", "uncorrected": "_DC_unc", "unmasked": "_DC"}

rows = []
for sample in dupcaller_idt_samples:
    sample_path = Path(dupcaller_path) / sample / "burden" / sample / f"{sample}_sbs_burden.txt"
    if not sample_path.exists():
        print(f"Warning: Burden file not found for sample {sample} at path {sample_path}")
        continue
    raw = pd.read_table(sample_path, sep="\t", header=None)[1]
    patient_id = sample.split("_")[0]
    for protocol, base_row in PROTOCOL_ROWS.items():
        rows.append({
            "sample": sample + PROTOCOL_SUFFIX[protocol],
            "protocol": protocol,
            "patient_id": patient_id,
            "mutrate_observed": raw.iloc[base_row],
            "mutrate_CI_low": raw.iloc[base_row + 1],
            "mutrate_CI_high": raw.iloc[base_row + 2],
        })

burden_df = pd.DataFrame(rows)
burden_df['caller'] = 'DupCaller'

# Generate tsv output
burden_df.to_csv("../data/dupcaller_burdens_cord_blood.tsv", sep = "\t", index=False)
